### 0. Environments

In [5]:
!pip install -q ir_datasets bm25s sentence-transformers faiss-gpu-cu11 ranx tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 23.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 44.8 MB/s eta 0:00:00


In [1]:
import os
import json
import time

from datetime import datetime
from pathlib import Path

import ir_datasets
import bm25s
import faiss
import numpy as np
import pandas as pd

from tqdm import tqdm
from itertools import islice
from sentence_transformers import SentenceTransformer
from ranx import Qrels, Run, evaluate


In [2]:
# =========================
# Configurations
# =========================
MAX_QUERIES: int = None          # số query dùng để đánh giá; đặt None để dùng toàn bộ dev/small
MAX_DOCS: int = 20_000          # hiện chưa dùng trong hàm load corpus bên dưới
TOP_K: int = 10               # nên >= max cutoff trong METRICS, ví dụ recall@100/map@100
BATCH_SIZE: int = 128

# BM25S query/runtime settings
# Lưu ý: nếu thay đổi BM25_TOKENIZE_KWARGS so với lúc build index,
# bạn cần rebuild lại sparse index để tokenization của corpus và query nhất quán.
BM25_BATCH_SIZE: int = 512
BM25_MMAP: bool = True
BM25_TOKENIZE_KWARGS: dict = {}

EMBEDDING_MODEL: str = "BAAI/bge-small-en-v1.5"
NORMALIZE_EMB: bool = True
DEVICE: str = "cuda"

# Dataset
CORPUS_ID: str = "msmarco-passage"
EVAL_ID: str = "msmarco-passage/dev"

# =========================
# Experiment output paths
# =========================
# Mỗi lần chạy notebook sẽ tạo một thư mục riêng để tránh ghi đè kết quả cũ.
RUN_ROOT: str = "/home/rmits/VDT-Hybrid-Search/runs"
RUN_NAME: str = "demo_run"
RUN_TIMESTAMP: str = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID: str = f"{RUN_NAME}_{RUN_TIMESTAMP}"

SAVE_PATH: str = os.path.join(RUN_ROOT, RUN_ID)
TREC_RUN_DIR: str = os.path.join(SAVE_PATH, "trec_runs") # lưu output dạng TREC run file để dễ dàng đánh giá với tools khác
QRELS_DIR: str = os.path.join(SAVE_PATH, "qrels")
METRICS_DIR: str = os.path.join(SAVE_PATH, "metrics")
CONFIG_DIR: str = os.path.join(SAVE_PATH, "configs")

for path in [SAVE_PATH, TREC_RUN_DIR, QRELS_DIR, METRICS_DIR, CONFIG_DIR]:
    os.makedirs(path, exist_ok=True)

print("RUN_ID:", RUN_ID)
print("SAVE_PATH:", SAVE_PATH)

# RRF fusion
RRF_K: int = 60

# Metrics thường dùng cho retrieval / MS MARCO style evaluation
METRICS = [
    "mrr@10",
    "ndcg@10",
    "precision@10",
    "recall@10",
    "recall@100",
    "map@100",
]

# Đường dẫn của các index đã tạo sẵn
SPARSE_INDEX_DIR: str = "/home/rmits/VDT-Hybrid-Search/bm25_index"
DENSE_INDEX_DIR: str = "/home/rmits/VDT-Hybrid-Search/bge_small_en_v1.5_embeddings_faiss"


RUN_ID: demo_run_20260612_081744
SAVE_PATH: /home/rmits/VDT-Hybrid-Search/runs/demo_run_20260612_081744


### 1. Download MSMARCO dataset

In [10]:
!cp -r /kaggle/input/datasets/hongkimgip/msmarco-passage/msmarco-passage /root/.ir_datasets/msmarco-passage

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [3]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, queries, qrels = download_and_preview_msmarco(
    corpus_id=CORPUS_ID,
    eval_id=EVAL_ID,
)



========== SAMPLE PASSAGES ==========

--- Sample Passage 1 ---
doc_id: 0
text: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

========== SAMPLE QUERIES ==========

--- Sample Querie 1 ---
query_id: 1048578
text: cost of endless pools/swim spa

========== SAMPLE QRELS ==========

--- Sample Qrel 1 ---
query_id: 1102432
doc_id: 2026790
relevance: 1

========== LOADING FULL CORPUS ==========


Loading passages: 8841823it [00:25, 352043.31it/s]


Total passages loaded: 8,841,823

========== LOADING QUERIES ==========


Loading queries: 101093it [00:00, 494942.51it/s]


Total queries loaded: 101,093

========== LOADING QRELS ==========


Loading qrels: 59273it [00:00, 562253.87it/s]

Total qrels queries loaded: 55,578


### 2. Retrieve

#### Sparse:

In [ ]:
class BM25SRetriever:
    def __init__(self, corpus=None, tokenize_kwargs=None):
        """
        Optimized BM25S retriever.

        Các điểm tối ưu chính:
        - Dùng cùng tokenize_kwargs cho cả build và search.
        - Load index bằng mmap=True cho index lớn.
        - Load doc_ids theo list tuần tự, tránh tạo dict row_to_doc_id lớn nếu không cần.
        - Tách raw retrieval khỏi bước convert sang dict để đo latency rõ hơn.
        - Hỗ trợ retrieve theo batch để tránh tokenize/retrieve toàn bộ eval set trong một lần quá lớn.
        """
        self.doc_ids = None
        self.doc_texts = None
        self.tokenize_kwargs = tokenize_kwargs or {}
        self.retriever = bm25s.BM25()

        if corpus is not None:
            # Cố định thứ tự doc_ids ngay từ đầu
            self.doc_ids = list(corpus.keys())
            self.doc_texts = [corpus[doc_id] for doc_id in self.doc_ids]

    def build(self):
        """
        Build BM25 index từ corpus.
        Chỉ gọi hàm này khi chưa có index local.
        """
        if self.doc_texts is None:
            raise ValueError("corpus is required to build BM25 index")

        corpus_tokens = bm25s.tokenize(
            self.doc_texts,
            **self.tokenize_kwargs,
        )
        self.retriever.index(corpus_tokens)

        print(f"Indexed {len(self.doc_texts):,} documents")

    def save(self, index_dir):
        """
        Save BM25 index + mapping row_id -> doc_id.
        """
        if self.doc_ids is None:
            raise ValueError("doc_ids is empty. Build from corpus before saving.")

        os.makedirs(index_dir, exist_ok=True)

        # Save BM25 index
        self.retriever.save(index_dir)

        # Save mapping
        doc_ids_path = os.path.join(index_dir, "doc_ids.jsonl")

        with open(doc_ids_path, "w", encoding="utf-8") as f:
            for row_id, doc_id in enumerate(self.doc_ids):
                f.write(json.dumps({
                    "row_id": row_id,
                    "doc_id": doc_id,
                }, ensure_ascii=False) + "\n")

        # Save lightweight config để nhắc lại tokenization setting.
        # Nếu tokenize_kwargs chứa object không serialize được, ví dụ stemmer,
        # phần này sẽ chỉ lưu string representation.
        config_path = os.path.join(index_dir, "bm25s_config.json")
        serializable_tokenize_kwargs = {
            key: str(value)
            for key, value in self.tokenize_kwargs.items()
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(
                {"tokenize_kwargs": serializable_tokenize_kwargs},
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(f"Saved BM25 index to: {index_dir}")
        print(f"Saved doc id mapping: {len(self.doc_ids):,}")

    @classmethod
    def load(cls, index_dir, mmap=True, tokenize_kwargs=None):
        """
        Load BM25 index + mapping từ local path.
        Không cần truyền corpus.

        tokenize_kwargs phải giống lúc build index.
        Nếu index cũ được build bằng bm25s.tokenize(self.doc_texts) mặc định,
        hãy để tokenize_kwargs=None hoặc {}.
        """
        obj = cls(corpus=None, tokenize_kwargs=tokenize_kwargs)

        # Load BM25 index
        obj.retriever = bm25s.BM25.load(index_dir, mmap=mmap)

        # Load mapping row_id -> doc_id
        # Tối ưu hơn cách tạo dict row_to_doc_id rồi convert lại sang list.
        doc_ids_path = os.path.join(index_dir, "doc_ids.jsonl")
        doc_ids = []

        with open(doc_ids_path, "r", encoding="utf-8") as f:
            for expected_row_id, line in enumerate(f):
                item = json.loads(line)
                row_id = int(item["row_id"])

                if row_id != expected_row_id:
                    raise ValueError(
                        "doc_ids.jsonl is not sequential. "
                        "Please sort by row_id before loading."
                    )

                doc_ids.append(str(item["doc_id"]))

        obj.doc_ids = doc_ids
        obj.doc_texts = None

        print(f"Loaded BM25 index from: {index_dir}")
        print(f"Loaded doc id mapping: {len(obj.doc_ids):,}")
        print(f"mmap: {mmap}")

        return obj

    def tokenize_queries(self, queries):
        """
        Tokenize queries bằng cùng setting với lúc build index.
        """
        if isinstance(queries, str):
            queries = [queries]

        return bm25s.tokenize(
            queries,
            **self.tokenize_kwargs,
        )

    def retrieve_tokens(self, query_tokens, top_k=100, return_dict=True, n_threads=0, chunk_size=128):
        """
        Retrieve từ query_tokens đã tokenize sẵn.

        return_dict=False dùng khi muốn đo riêng pure retrieval time.
        return_dict=True trả về list[{doc_id: score}] để tương thích với ranx/fusion.
        """
        if self.doc_ids is None:
            raise ValueError("doc_ids is empty. Build or load index first.")

        results, scores = self.retriever.retrieve(
            query_tokens,
            corpus=self.doc_ids,
            k=top_k,
            n_threads=n_threads,
            chunksize=chunk_size,
        )

        if not return_dict:
            return results, scores

        return self._format_results(results, scores)

    def search(self, queries, top_k=100, return_dict=True, n_threads=0, chunk_size=128):
        """
        Search một batch queries.

        Với evaluation lớn, nên dùng search_batched(...) để tránh batch quá lớn
        và để có latency stats theo batch.
        """
        query_tokens = self.tokenize_queries(queries)
        return self.retrieve_tokens(
            query_tokens=query_tokens,
            top_k=top_k,
            return_dict=return_dict,
            n_threads=n_threads,
            chunk_size=chunk_size,
        )

    def search_batched(self, queries, top_k=100, batch_size=512, show_progress=True):
        """
        Search theo batch để tối ưu throughput và kiểm soát memory.

        Trả về:
            all_results: list[{doc_id: score}]
            stats: dict latency/qps
        """
        if isinstance(queries, str):
            queries = [queries]

        all_results = []
        batch_times = []

        iterator = range(0, len(queries), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="BM25S batched search")

        for start_idx in iterator:
            batch_queries = queries[start_idx:start_idx + batch_size]

            start = time.perf_counter()
            batch_results = self.search(
                batch_queries,
                top_k=top_k,
                return_dict=True,
            )
            elapsed = time.perf_counter() - start

            all_results.extend(batch_results)
            batch_times.append({
                "batch_size": len(batch_queries),
                "seconds": elapsed,
                "seconds_per_query": elapsed / len(batch_queries),
            })

        stats = self._latency_stats_from_batches(batch_times)
        return all_results, stats

    @staticmethod
    def _format_results(results, scores):
        """
        Convert raw bm25s outputs sang list[dict].
        Chỉ gọi sau khi đã đo pure retrieval nếu cần benchmark chính xác.
        """
        all_results = []

        for doc_id_list, score_list in zip(results, scores):
            run = {
                str(doc_id): float(score)
                for doc_id, score in zip(doc_id_list, score_list)
            }
            all_results.append(run)

        return all_results

    @staticmethod
    def _latency_stats_from_batches(batch_times):
        if not batch_times:
            return {
                "total_seconds": 0.0,
                "num_queries": 0,
                "avg_latency_seconds_per_query": 0.0,
                "qps": 0.0,
                "p50_batch_latency_ms_per_query": 0.0,
                "p90_batch_latency_ms_per_query": 0.0,
                "p95_batch_latency_ms_per_query": 0.0,
                "p99_batch_latency_ms_per_query": 0.0,
            }

        total_seconds = sum(item["seconds"] for item in batch_times)
        num_queries = sum(item["batch_size"] for item in batch_times)
        per_query = np.array(
            [item["seconds_per_query"] for item in batch_times],
            dtype="float64",
        )

        return {
            "total_seconds": float(total_seconds),
            "num_queries": int(num_queries),
            "avg_latency_seconds_per_query": float(total_seconds / num_queries),
            "qps": float(num_queries / total_seconds) if total_seconds > 0 else 0.0,
            # Đây là pXX trên các batch latency/query, không phải true per-query latency.
            "p50_batch_latency_ms_per_query": float(np.percentile(per_query, 50) * 1000),
            "p90_batch_latency_ms_per_query": float(np.percentile(per_query, 90) * 1000),
            "p95_batch_latency_ms_per_query": float(np.percentile(per_query, 95) * 1000),
            "p99_batch_latency_ms_per_query": float(np.percentile(per_query, 99) * 1000),
        }


bm25_retriever = BM25SRetriever.load(
    SPARSE_INDEX_DIR,
    mmap=BM25_MMAP,
    tokenize_kwargs=BM25_TOKENIZE_KWARGS,
)


Loaded BM25 index from: /home/rmits/VDT-Hybrid-Search/bm25_index
Loaded doc id mapping: 8,841,823
mmap: True


Search thử với BM25S optimized retriever:


In [13]:
query_ids = list(queries.keys())[:5]
query_texts = [queries[qid] for qid in query_ids]
bm25_results = bm25_retriever.search(query_texts, top_k=10, n_threads=4, chunk_size=128)

for qid, qtext, result in zip(query_ids, query_texts, bm25_results):
    print("=" * 80)
    print("QID:", qid)
    print("Query:", qtext)
    for doc_id, score in list(result.items())[:5]:
        print(score, '|', doc_id, '|', corpus[doc_id][:200])

Split strings:   0%|          | 0/5 [00:00<?, ?it/s]

TypeError: BM25.retrieve() got an unexpected keyword argument 'chunk_size'

#### Dense:

In [7]:
class DenseFaissRetriever:
    def __init__(
        self,
        corpus=None,
        model_name="BAAI/bge-small-en-v1.5",
        batch_size=128,
        device=None,
        normalize_embeddings=True,
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = device
        self.normalize_embeddings = normalize_embeddings

        self.doc_ids = None
        self.doc_texts = None
        self.row_id_to_doc_id = None

        if corpus is not None:
            # Cố định thứ tự doc_ids
            self.doc_ids = list(corpus.keys())
            self.doc_texts = [corpus[doc_id] for doc_id in self.doc_ids]

            self.row_id_to_doc_id = {
                row_id: doc_id
                for row_id, doc_id in enumerate(self.doc_ids)
            }

        self.model = SentenceTransformer(model_name, device=device)

        self.index = None
        self.embeddings = None

    def build(self):
        """
        Build FAISS index từ corpus.
        Chỉ gọi hàm này khi chưa có index local.
        """
        if self.doc_texts is None:
            raise ValueError("corpus is required to build FAISS index")

        embeddings = self.model.encode(
            self.doc_texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize_embeddings,
        ).astype("float32")

        dim = embeddings.shape[1]

        if self.normalize_embeddings:
            base_index = faiss.IndexFlatIP(dim)
            index_type = "IndexFlatIP"
        else:
            base_index = faiss.IndexFlatL2(dim)
            index_type = "IndexFlatL2"

        # Dùng IDMap2 để FAISS trả về row_id đúng với metadata
        index = faiss.IndexIDMap2(base_index)

        row_ids = np.arange(len(self.doc_ids)).astype("int64")
        index.add_with_ids(embeddings, row_ids)

        self.embeddings = embeddings
        self.index = index

        self.row_id_to_doc_id = {
            row_id: doc_id
            for row_id, doc_id in enumerate(self.doc_ids)
        }

        assert self.index.ntotal == len(self.doc_ids)

        print("FAISS index size:", self.index.ntotal)
        print("Embedding dim:", dim)
        print("Index type:", index_type)

    def save(
        self,
        index_dir,
        save_embeddings=False,
        faiss_filename="faiss.index",
        metadata_filename="doc_ids.jsonl",
        embeddings_filename="embeddings.npy",
        config_filename="config.json",
    ):
        """
        Save FAISS index + metadata mapping row_id -> doc_id.
        File name mặc định khớp với code bạn đang dùng:
            - faiss.index
            - doc_ids.jsonl
            - embeddings.npy
        """
        if self.index is None:
            raise ValueError("FAISS index is empty. Build index before saving.")

        if self.doc_ids is None:
            raise ValueError("doc_ids is empty. Build from corpus before saving.")

        os.makedirs(index_dir, exist_ok=True)

        faiss_index_path = os.path.join(index_dir, faiss_filename)
        metadata_path = os.path.join(index_dir, metadata_filename)
        config_path = os.path.join(index_dir, config_filename)

        faiss.write_index(self.index, faiss_index_path)

        with open(metadata_path, "w", encoding="utf-8") as f:
            for row_id, doc_id in enumerate(self.doc_ids):
                f.write(json.dumps({
                    "row_id": row_id,
                    "doc_id": doc_id,
                }, ensure_ascii=False) + "\n")

        config = {
            "model_name": self.model_name,
            "batch_size": self.batch_size,
            "normalize_embeddings": self.normalize_embeddings,
            "num_docs": len(self.doc_ids),
            "dim": self.index.d,
            "faiss_filename": faiss_filename,
            "metadata_filename": metadata_filename,
        }

        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, ensure_ascii=False, indent=2)

        if save_embeddings and self.embeddings is not None:
            embeddings_path = os.path.join(index_dir, embeddings_filename)
            np.save(embeddings_path, self.embeddings)
            print(f"Saved embeddings to: {embeddings_path}")

        print(f"Saved FAISS index to: {faiss_index_path}")
        print(f"Saved metadata to: {metadata_path}")
        print(f"Saved config to: {config_path}")

    @classmethod
    def load(
        cls,
        index_dir,
        model_name="BAAI/bge-small-en-v1.5",
        batch_size=128,
        device=None,
        normalize_embeddings=True,
        faiss_filename="faiss.index",
        metadata_filename="doc_ids.jsonl",
    ):
        """
        Load FAISS index + metadata từ local path.
        Không cần truyền corpus.

        Yêu cầu trong index_dir có:
            - faiss.index
            - doc_ids.jsonl
        """
        faiss_index_path = os.path.join(index_dir, faiss_filename)
        metadata_path = os.path.join(index_dir, metadata_filename)

        if not os.path.exists(faiss_index_path):
            raise FileNotFoundError(f"FAISS index not found: {faiss_index_path}")

        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"Metadata not found: {metadata_path}")

        obj = cls(
            corpus=None,
            model_name=model_name,
            batch_size=batch_size,
            device=device,
            normalize_embeddings=normalize_embeddings,
        )

        obj.index = faiss.read_index(faiss_index_path)

        row_id_to_doc_id = {}

        with open(metadata_path, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                row_id = int(item["row_id"])
                doc_id = item["doc_id"]
                row_id_to_doc_id[row_id] = doc_id

        obj.row_id_to_doc_id = row_id_to_doc_id

        # Chỉ để tiện truy cập tuần tự; search sẽ dùng row_id_to_doc_id
        obj.doc_ids = [
            row_id_to_doc_id[i]
            for i in range(len(row_id_to_doc_id))
        ]

        obj.doc_texts = None
        obj.embeddings = None

        if obj.index.ntotal != len(obj.row_id_to_doc_id):
            raise ValueError(
                f"FAISS index size != metadata size: "
                f"{obj.index.ntotal} != {len(obj.row_id_to_doc_id)}"
            )

        print(f"Loaded FAISS index from: {faiss_index_path}")
        print(f"Loaded metadata from: {metadata_path}")
        print(f"Loaded doc id mapping: {len(obj.row_id_to_doc_id):,}")
        print(f"FAISS index size: {obj.index.ntotal:,}")

        return obj

    def search(self, queries, top_k=100):
        """
        Return format:
            [
                {
                    doc_id: score,
                    ...
                },
                ...
            ]
        """
        if self.index is None:
            raise ValueError("FAISS index is empty. Build or load index first.")

        if self.row_id_to_doc_id is None:
            raise ValueError("row_id_to_doc_id is empty. Build or load metadata first.")

        if isinstance(queries, str):
            queries = [queries]

        query_embeddings = self.model.encode(
            queries,
            batch_size=self.batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize_embeddings,
        ).astype("float32")

        scores, row_ids = self.index.search(query_embeddings, top_k)

        all_results = []

        for score_list, row_id_list in zip(scores, row_ids):
            run = {}

            for score, row_id in zip(score_list, row_id_list):
                if row_id == -1:
                    continue

                row_id = int(row_id)

                if row_id not in self.row_id_to_doc_id:
                    raise KeyError(
                        f"row_id={row_id} returned by FAISS but not found in metadata"
                    )

                doc_id = self.row_id_to_doc_id[row_id]
                run[str(doc_id)] = float(score)

            all_results.append(run)

        return all_results

dense_retriever = DenseFaissRetriever.load(
    index_dir=DENSE_INDEX_DIR,
    model_name=EMBEDDING_MODEL,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    normalize_embeddings=NORMALIZE_EMB,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded FAISS index from: /home/rmits/VDT-Hybrid-Search/bge_small_en_v1.5_embeddings_faiss/faiss.index
Loaded metadata from: /home/rmits/VDT-Hybrid-Search/bge_small_en_v1.5_embeddings_faiss/doc_ids.jsonl
Loaded doc id mapping: 8,841,823
FAISS index size: 8,841,823


Test:

In [8]:
query_ids = list(queries.keys())[:5]
query_texts = [queries[qid] for qid in query_ids]
dense_results = dense_retriever.search(query_texts, top_k=10)

for qid, qtext, result in zip(query_ids, query_texts, dense_results):
    print("=" * 80)
    print("QID:", qid)
    print("Query:", qtext)
    for doc_id, score in list(result.items())[:3]:
        print(score, doc_id, corpus[doc_id][:200])

QID: 1048578
Query: cost of endless pools/swim spa
0.9119080305099487 7187234 Endless pools and swim spas are available in a number of different price brackets. For the brand called “Endless Pools” prices start at $23,900 for their most basic pool For other brands of endless po
0.8883839845657349 7187239 Endless Pools offers a family of swimming machines and therapy pools starting as low as $7,400. Endless Pools Fitness Systems start at $19,999. Each model comes standard with three hydromassage seats,
0.88593590259552 4567130 $23,900. The 17' Endless Pool Spa features a larger exercise area that is ideal for swimming, aquatic exercise and fun. With easy access full depth stairs and up to two jet seats with over 20 jets you
QID: 1048579
Query: what is pcnt
0.7222236394882202 7187232 1 meanings of PCNT acronym and PCNT abbreviation. Get the Medical definition of PCNT by All Acronyms dictionary. Top Definition: Pericentrin In Medical dictionary category.
0.712819516658783 7187227 PCNT sta

### 3. Fusion

In [10]:
def rrf_fusion_one(sparse_run, dense_run, k=60, top_k=100):
    fused = {}

    for rank, doc_id in enumerate(sparse_run.keys(), start=1):
        fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)

    for rank, doc_id in enumerate(dense_run.keys(), start=1):
        fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)

    fused = dict(
        sorted(
            fused.items(),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]
    )

    return fused


def rrf_fusion_all(sparse_results, dense_results, k=60, top_k=100):
    fused_results = []

    for sparse_run, dense_run in zip(sparse_results, dense_results):
        fused_results.append(
            rrf_fusion_one(
                sparse_run=sparse_run,
                dense_run=dense_run,
                k=k,
                top_k=top_k,
            )
        )

    return fused_results

#### Full retrieval:

In [11]:
# =========================
# Full retrieval for evaluation
# =========================

def prepare_eval_queries(queries, qrels, max_queries=None):
    """
    Chỉ đánh giá trên các query có qrels.

    Args:
        queries: dict {query_id: query_text}
        qrels: dict {query_id: {doc_id: relevance}}
        max_queries: số query tối đa, None để dùng toàn bộ

    Returns:
        eval_query_ids: list[str]
        eval_query_texts: list[str]
    """
    eval_query_ids = [
        str(qid)
        for qid in queries.keys()
        if qid in qrels and len(qrels[qid]) > 0
    ]

    if max_queries is not None:
        eval_query_ids = eval_query_ids[:max_queries]

    eval_query_texts = [queries[qid] for qid in eval_query_ids]

    print(f"Number of eval queries: {len(eval_query_ids):,}")
    return eval_query_ids, eval_query_texts


def _latency_stats(total_seconds, num_queries):
    """
    Summary thời gian chạy ở mức toàn bộ batch.
    Lưu ý: đây là average latency theo total_time/num_queries.
    """
    avg_latency = total_seconds / num_queries if num_queries else 0.0
    qps = num_queries / total_seconds if total_seconds > 0 else 0.0

    return {
        "total_seconds": float(total_seconds),
        "num_queries": int(num_queries),
        "avg_latency_seconds_per_query": float(avg_latency),
        "qps": float(qps),
    }


def _merge_latency_stats(base_stats, extra_stats):
    """
    Gộp stats đơn giản để giữ cùng schema lưu output.
    """
    if not extra_stats:
        return base_stats

    merged = dict(base_stats)
    merged.update(extra_stats)
    return merged


def retrieval_for_eval(query_texts, top_k=100):
    """
    Chạy 3 hệ retrieval:
        - BM25 sparse
        - Dense FAISS
        - Hybrid RRF

    BM25 đã được tối ưu:
        - retrieve theo batch
        - tách stats theo batch
        - tránh tokenize toàn bộ eval set trong một call quá lớn
    """
    num_queries = len(query_texts)

    # BM25S batched retrieval
    bm25_results, bm25_batch_stats = bm25_retriever.search_batched(
        query_texts,
        top_k=top_k,
        batch_size=BM25_BATCH_SIZE,
        show_progress=True,
    )
    bm25_time = bm25_batch_stats["total_seconds"]

    # Dense retrieval hiện vẫn dùng implementation cũ.
    # Có thể tối ưu tương tự bằng cách encode/search theo batch nếu cần.
    start = time.perf_counter()
    dense_results = dense_retriever.search(query_texts, top_k=top_k)
    dense_time = time.perf_counter() - start

    start = time.perf_counter()
    hybrid_results = rrf_fusion_all(
        sparse_results=bm25_results,
        dense_results=dense_results,
        k=RRF_K,
        top_k=top_k,
    )
    hybrid_time = time.perf_counter() - start

    retrieval_stats = {
        "bm25s": _merge_latency_stats(
            _latency_stats(bm25_time, num_queries),
            bm25_batch_stats,
        ),
        "dense_faiss": _latency_stats(dense_time, num_queries),
        "hybrid_rrf_fusion_only": _latency_stats(hybrid_time, num_queries),
        "hybrid_rrf_end_to_end": _latency_stats(bm25_time + dense_time + hybrid_time, num_queries),
    }

    print(f"BM25 retrieval time:        {bm25_time:.2f}s")
    print(f"Dense retrieval time:       {dense_time:.2f}s")
    print(f"RRF fusion time:            {hybrid_time:.2f}s")
    print(f"Hybrid end-to-end time:     {bm25_time + dense_time + hybrid_time:.2f}s")
    print(f"BM25 QPS:                   {retrieval_stats['bm25s']['qps']:.2f}")

    return bm25_results, dense_results, hybrid_results, retrieval_stats


eval_query_ids, eval_query_texts = prepare_eval_queries(
    queries=queries,
    qrels=qrels,
    max_queries=MAX_QUERIES,
)

bm25_results, dense_results, hybrid_results, retrieval_stats = retrieval_for_eval(
    query_texts=eval_query_texts,
    top_k=TOP_K,
)


Number of eval queries: 55,578


BM25S batched search:   0%|          | 0/109 [00:00<?, ?it/s]

Split strings:   0%|          | 0/512 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/512 [00:00<?, ?it/s]

BM25S batched search:   1%|          | 1/109 [04:43<8:31:04, 283.93s/it]

...

BM25S batched search:   3%|▎         | 3/109 [14:05<8:17:45, 281.75s/it]

Split strings:   0%|          | 0/512 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/512 [00:00<?, ?it/s]

BM25S batched search:   3%|▎         | 3/109 [14:09<8:20:05, 283.07s/it]


KeyboardInterrupt: 

In [18]:
empty_queries = []

for qid in queries.keys():
    reldocs = qrels.get(qid, None)
    if not reldocs:
        empty_queries.append(qid)

len(empty_queries)

45515

### 4. Evaluate & Save Outputs

Đánh giá BM25, Dense FAISS và Hybrid RRF bằng `ranx` trên cùng một tập query, sau đó lưu lại:

- TREC run files cho từng retriever
- TREC qrels cho tập query evaluate
- metric summary dạng CSV/JSON
- config của lần chạy
- manifest tổng hợp đường dẫn artifact


In [14]:
# =========================
# Evaluation helpers
# =========================

def make_run_dict(query_ids, results):
    """
    Convert output của retriever sang format ranx / TREC-friendly:
        {qid: {doc_id: score}}
    """
    if len(query_ids) != len(results):
        raise ValueError(
            f"len(query_ids) != len(results): {len(query_ids)} != {len(results)}"
        )

    return {
        str(qid): {
            str(doc_id): float(score)
            for doc_id, score in result.items()
        }
        for qid, result in zip(query_ids, results)
    }


def make_qrels_dict(query_ids, qrels, keep_zero_relevance=False):
    """
    Convert qrels sang format ranx / TREC-friendly:
        {qid: {doc_id: relevance}}

    Mặc định chỉ giữ relevance > 0 vì retrieval metrics thường xem doc rel=0 là non-relevant.
    Khi lưu TREC qrels, có thể đặt keep_zero_relevance=True để giữ format đầy đủ hơn.
    """
    qrels_out = {}

    for qid in query_ids:
        qid = str(qid)
        rel_docs = qrels.get(qid, {})

        filtered = {
            str(doc_id): int(rel)
            for doc_id, rel in rel_docs.items()
            if keep_zero_relevance or int(rel) > 0
        }

        if filtered:
            qrels_out[qid] = filtered

    return qrels_out


def build_ranx_objects(query_ids, qrels, bm25_results, dense_results, hybrid_results):
    qrels_obj = Qrels(make_qrels_dict(query_ids, qrels))

    runs = {
        "BM25S": Run(make_run_dict(query_ids, bm25_results), name="BM25S"),
        "Dense FAISS": Run(make_run_dict(query_ids, dense_results), name="Dense FAISS"),
        "Hybrid RRF": Run(make_run_dict(query_ids, hybrid_results), name="Hybrid RRF"),
    }

    return qrels_obj, runs


def evaluate_runs(qrels_obj, runs, metrics):
    rows = []

    for run_name, run in runs.items():
        scores = evaluate(qrels_obj, run, metrics)
        rows.append({"run": run_name, **scores})

    scores_df = pd.DataFrame(rows).set_index("run")
    return scores_df


def qrels_coverage_report(query_ids, qrels, *retrievers):
    """
    Kiểm tra có bao nhiêu relevant docs nằm trong index.
    Hữu ích khi bạn đang dùng demo index / subset corpus.
    """
    indexed_sets = []
    for retriever in retrievers:
        if getattr(retriever, "doc_ids", None) is not None:
            indexed_sets.append(set(map(str, retriever.doc_ids)))

    if not indexed_sets:
        print("No retriever doc_ids found for coverage report.")
        return None

    # Nếu đánh giá hybrid, doc được retrieve bởi cả hai hệ nên intersection là corpus công bằng nhất.
    common_indexed_doc_ids = set.intersection(*indexed_sets)

    total_rel = 0
    covered_rel = 0
    queries_with_covered_rel = 0

    for qid in query_ids:
        rel_docs = {
            str(doc_id)
            for doc_id, rel in qrels.get(str(qid), {}).items()
            if int(rel) > 0
        }
        total_rel += len(rel_docs)
        covered = rel_docs & common_indexed_doc_ids
        covered_rel += len(covered)
        queries_with_covered_rel += int(len(covered) > 0)

    report = {
        "eval_queries": len(query_ids),
        "queries_with_relevant_doc_in_index": queries_with_covered_rel,
        "total_relevant_docs": total_rel,
        "relevant_docs_in_common_index": covered_rel,
        "relevant_doc_coverage": covered_rel / total_rel if total_rel else 0.0,
    }

    return pd.DataFrame([report])


# =========================
# Saving helpers: TREC run, TREC qrels, metrics, config
# =========================

def _json_safe(obj):
    """Convert numpy / pandas scalar types to JSON-serializable Python types."""
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(_json_safe(v) for v in obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if obj is None:
        return None
    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass
    return obj


def save_json(obj, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(_json_safe(obj), f, ensure_ascii=False, indent=2)


def save_trec_run(run_dict, output_path, run_name="run"):
    """
    Save retrieval output theo TREC run format:
        query_id Q0 doc_id rank score run_name

    Args:
        run_dict: {qid: {doc_id: score}}
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        for qid, doc_scores in run_dict.items():
            sorted_docs = sorted(
                doc_scores.items(),
                key=lambda x: x[1],
                reverse=True,
            )

            for rank, (doc_id, score) in enumerate(sorted_docs, start=1):
                f.write(f"{qid} Q0 {doc_id} {rank} {float(score):.8f} {run_name}\n")


def save_trec_qrels(qrels_dict, output_path):
    """
    Save ground truth theo TREC qrels format:
        query_id 0 doc_id relevance

    Args:
        qrels_dict: {qid: {doc_id: relevance}}
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        for qid, doc_rels in qrels_dict.items():
            for doc_id, rel in doc_rels.items():
                f.write(f"{qid} 0 {doc_id} {int(rel)}\n")


def save_experiment_artifacts(
    query_ids,
    qrels,
    run_dicts,
    scores_df,
    coverage_df=None,
    retrieval_stats=None,
):
    """
    Lưu toàn bộ artifact của một lần chạy:
        - TREC run cho từng retriever
        - TREC qrels cho tập query đang evaluate
        - metrics summary CSV/JSON
        - per-run metrics JSON
        - coverage report
        - experiment config
        - summary manifest
    """
    for path in [TREC_RUN_DIR, QRELS_DIR, METRICS_DIR, CONFIG_DIR, SAVE_PATH]:
        Path(path).mkdir(parents=True, exist_ok=True)

    run_paths = {}

    for run_name, run_dict in run_dicts.items():
        output_path = Path(TREC_RUN_DIR) / f"{run_name}.trec"
        save_trec_run(run_dict, output_path, run_name=run_name)
        run_paths[run_name] = str(output_path)

    qrels_for_save = make_qrels_dict(
        query_ids=query_ids,
        qrels=qrels,
        keep_zero_relevance=True,
    )
    qrels_path = Path(QRELS_DIR) / f"{EVAL_ID.replace('/', '_')}.qrels"
    save_trec_qrels(qrels_for_save, qrels_path)

    metrics_csv_path = Path(METRICS_DIR) / "metrics_summary.csv"
    metrics_json_path = Path(METRICS_DIR) / "metrics_summary.json"
    scores_df.to_csv(metrics_csv_path)
    save_json(scores_df.to_dict(orient="index"), metrics_json_path)

    per_run_metric_paths = {}
    for run_name in run_dicts.keys():
        # mapping tên file gọn sang tên index trong scores_df
        score_index = {
            "bm25s": "BM25S",
            "dense_faiss": "Dense FAISS",
            "hybrid_rrf": "Hybrid RRF",
        }.get(run_name, run_name)

        if score_index in scores_df.index:
            per_run_path = Path(METRICS_DIR) / f"{run_name}_metrics.json"
            save_json(scores_df.loc[score_index].to_dict(), per_run_path)
            per_run_metric_paths[run_name] = str(per_run_path)

    coverage_path = None
    if coverage_df is not None:
        coverage_path = Path(METRICS_DIR) / "qrels_coverage.csv"
        coverage_df.to_csv(coverage_path, index=False)

    config = {
        "run_id": RUN_ID,
        "run_name": RUN_NAME,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "corpus_id": CORPUS_ID,
        "eval_id": EVAL_ID,
        "max_queries": MAX_QUERIES,
        "num_eval_queries": len(query_ids),
        "top_k": TOP_K,
        "metrics": METRICS,
        "sparse_index_dir": SPARSE_INDEX_DIR,
        "dense_index_dir": DENSE_INDEX_DIR,
        "embedding_model": EMBEDDING_MODEL,
        "normalize_embeddings": NORMALIZE_EMB,
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "fusion_method": "rrf",
        "rrf_k": RRF_K,
    }

    config_path = Path(CONFIG_DIR) / "run_config.json"
    save_json(config, config_path)

    manifest = {
        "run_id": RUN_ID,
        "save_path": SAVE_PATH,
        "trec_run_paths": run_paths,
        "qrels_path": str(qrels_path),
        "metrics_csv_path": str(metrics_csv_path),
        "metrics_json_path": str(metrics_json_path),
        "per_run_metric_paths": per_run_metric_paths,
        "coverage_path": str(coverage_path) if coverage_path else None,
        "config_path": str(config_path),
        "retrieval_stats": retrieval_stats or {},
        "scores": scores_df.to_dict(orient="index"),
    }

    manifest_path = Path(SAVE_PATH) / "manifest.json"
    save_json(manifest, manifest_path)

    artifacts_df = pd.DataFrame([
        {"artifact": "save_path", "path": SAVE_PATH},
        {"artifact": "qrels", "path": str(qrels_path)},
        {"artifact": "metrics_csv", "path": str(metrics_csv_path)},
        {"artifact": "metrics_json", "path": str(metrics_json_path)},
        {"artifact": "coverage", "path": str(coverage_path) if coverage_path else None},
        {"artifact": "config", "path": str(config_path)},
        {"artifact": "manifest", "path": str(manifest_path)},
        *[
            {"artifact": f"trec_run:{name}", "path": path}
            for name, path in run_paths.items()
        ],
    ])

    print(f"Saved experiment artifacts to: {SAVE_PATH}")
    return artifacts_df, manifest


In [15]:
qrels_obj, runs = build_ranx_objects(
    query_ids=eval_query_ids,
    qrels=qrels,
    bm25_results=bm25_results,
    dense_results=dense_results,
    hybrid_results=hybrid_results,
)

scores_df = evaluate_runs(
    qrels_obj=qrels_obj,
    runs=runs,
    metrics=METRICS,
)

display(scores_df)

coverage_df = qrels_coverage_report(
    eval_query_ids,
    qrels,
    bm25_retriever,
    dense_retriever,
)

display(coverage_df)

# =========================
# Save outputs for this run
# =========================
run_dicts = {
    "bm25s": make_run_dict(eval_query_ids, bm25_results),
    "dense_faiss": make_run_dict(eval_query_ids, dense_results),
    "hybrid_rrf": make_run_dict(eval_query_ids, hybrid_results),
}

artifacts_df, manifest = save_experiment_artifacts(
    query_ids=eval_query_ids,
    qrels=qrels,
    run_dicts=run_dicts,
    scores_df=scores_df,
    coverage_df=coverage_df,
    retrieval_stats=retrieval_stats,
)

display(artifacts_df)


,mrr@10,ndcg@10,precision@10,recall@10,recall@100,map@100
run,,,,,,
BM25S,0.188675,0.232415,0.0395,0.3750,0.660000,0.197266
Dense FAISS,0.351446,0.404936,0.0625,0.6000,0.868333,0.354117
Hybrid RRF,0.309921,0.360921,0.0555,0.5275,0.856667,0.323023


,eval_queries,queries_with_relevant_doc_in_index,total_relevant_docs,relevant_docs_in_common_index,relevant_doc_coverage
0,200,200,212,212,1.0


Saved experiment artifacts to: /home/rmits/VDT-Hybrid-Search/runs/demo_run_20260611_134702


,artifact,path
0,save_path,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
1,qrels,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
2,metrics_csv,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
3,metrics_json,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
4,coverage,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
5,config,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
6,manifest,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
7,trec_run:bm25s,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
8,trec_run:dense_faiss,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
9,trec_run:hybrid_rrf,/home/rmits/VDT-Hybrid-Search/runs/demo_run_20...
